# 实验一：静态页面与 API 数据采集

**平台**：SmartPark DataLab（`https://115.159.159.82`，教学环境，使用自签名证书）

本实验完成两类典型 Web 数据采集：

```text
Device  ：动态加载 → 找到 API → JSON → 分页采集全部设备（5000 条）
Company ：静态加载 → HTML → BeautifulSoup → "下一页" → 采集 3 页（60 条）
```

> 说明：站点使用自签名 HTTPS 证书，因此请求时设置 `verify=False` 并关闭对应警告。

In [1]:
import json
import warnings
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
import urllib3
from bs4 import BeautifulSoup

# 教学环境为自签名证书，跳过证书校验并屏蔽警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://115.159.159.82"
session = requests.Session()
session.verify = False
print("requests", requests.__version__)

requests 2.34.2


## 任务一：判断数据加载方式

判断方法：在页面上找一条真实显示的数据，然后 `右键 → 查看网页源代码`，
`Ctrl + F` 搜索这条数据：

- 源代码中**能找到** → 数据随 HTML 一起返回，属于**静态加载**；
- 源代码中**找不到** → 数据由 JS 异步获取后渲染，属于**动态加载**，
  再通过 `F12 → Network → Fetch/XHR` 找到返回数据的 API。

In [2]:
# Device 页面：页面加载后能看到设备名"循环泵-000001-001"，检查它是否在初始 HTML 中
r = session.get(BASE_URL + "/devices", timeout=10)
r.raise_for_status()
devices_html = r.text

probe = "循环泵-000001-001"
print("源代码中搜索设备名:", repr(probe))
print("搜索结果:", "找到" if probe in devices_html else "未找到")
print("结论：设备数据不在初始 HTML 中，/devices 页面为【动态加载】")

源代码中搜索设备名: '循环泵-000001-001'
搜索结果: 未找到
结论：设备数据不在初始 HTML 中，/devices 页面为【动态加载】


在 Network 的 Fetch/XHR 中刷新页面，可以找到返回设备列表的请求：

```text
Request URL   : https://115.159.159.82/api/devices
Request Method: GET
Query 参数    : page=1, page_size=50
Response      : JSON，包含 page / page_size / total / items 四个字段
```

（截图 1：`images/01_device_api.png`）

## 任务二：Device API 分页采集

先用 requests 复现第一页请求，观察 JSON 结构。

In [3]:
api_url = BASE_URL + "/api/devices"

r = session.get(api_url, params={"page": 1, "page_size": 50}, timeout=10)
r.raise_for_status()

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [4]:
data = r.json()

print(type(data))
print("keys:", list(data.keys()))

<class 'dict'>
keys: ['page', 'page_size', 'total', 'items']


In [5]:
print("当前页 page        =", data["page"])
print("每页条数 page_size =", data["page_size"])
print("设备总数 total     =", data["total"])
print("本页设备列表 items =", len(data["items"]), "条")
print()
print("第一条设备记录：")
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))

当前页 page        = 1
每页条数 page_size = 50
设备总数 total     = 5000
本页设备列表 items = 50 条

第一条设备记录：
{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


JSON 中四个分页字段已经明确：`page`（当前页）、`page_size`（每页条数）、
`total`（设备总数）、`items`（本页设备列表）。

总页数由 `total` 和 `page_size` 计算得到，而不是在代码里写死——
这样即使接口数据量变化，程序仍然正确。

In [6]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [7]:
devices = []

for page in range(1, total_pages + 1):
    r = session.get(
        api_url,
        params={"page": page, "page_size": page_size},
        timeout=10
    )
    r.raise_for_status()

    page_data = r.json()
    devices.extend(page_data["items"])

    if page <= 2 or page == total_pages:
        print(f"第 {page} 页完成，累计 {len(devices)} 条")

print("device count:", len(devices))

第 1 页完成，累计 50 条


第 2 页完成，累计 100 条


第 100 页完成，累计 5000 条
device count: 5000


## 任务三：Company 静态页面解析

`/companies` 页面：`右键 → 查看网页源代码`，`Ctrl + F` 搜索页面上真实显示的
企业名称（如"临港精工001有限公司"），可以找到，说明企业数据随 HTML 返回，
属于**静态加载**（截图 2：`images/02_company_source.png`）。

### 企业记录 Selector 设计

在 Elements 中观察，一条完整企业记录的最外层节点是：

```html
<article class="company-card" data-company-id="C000001"> ... </article>
```

按"id → class → data-* → 层级"的顺序考虑：

- 没有 id；`class="company-card"` 语义明确、全页唯一专用 → 选它；
- `data-company-id` 属性正好携带 `company_id`；
- 最终 selector：`article.company-card`，短且稳定。

In [8]:
base_url = BASE_URL
current_url = base_url + "/companies"

r = session.get(current_url, timeout=10)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

# 先在页面显示的企业名中取一个，验证它确实写在源代码里
name_on_page = soup.select_one("article.company-card h2 a").get_text(strip=True)
print("页面上显示的企业名:", name_on_page)
print("源代码中搜索结果:", "找到" if name_on_page in r.text else "未找到")
print("结论：企业数据随 HTML 返回，/companies 页面为【静态加载】")

页面上显示的企业名: 临港精工001有限公司
源代码中搜索结果: 找到
结论：企业数据随 HTML 返回，/companies 页面为【静态加载】


In [9]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


先定位一条记录，再在记录内部提取字段。企业名、状态徽标是文本；
`company_id` 是 `data-*` 属性；行业、区域、设备数量在 `dl > div > (dt, dd)` 结构里。

In [10]:
card = cards[0]

# data-* 属性：企业编号
company_id = card.get("data-company-id")

# 文本：企业名（h2 内的 a）、状态徽标
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge").get_text(strip=True)

# dt/dd 结构：先转换成字典，再按中文键取出
details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print("company_id  :", company_id)
print("name        :", name)
print("status      :", status)
print("industry    :", industry)
print("zone        :", zone)
print("device_count:", device_count)

company_id  : C000001
name        : 临港精工001有限公司
status      : 运营中
industry    : 汽车零部件
zone        : 智造一区
device_count: 6


把提取逻辑封装成函数，第一页 20 条记录循环提取。

In [11]:
def parse_company(card):
    """从一条企业记录（article.company-card）中提取字段"""
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),
    }


companies = [parse_company(card) for card in soup.select("article.company-card")]

print("本页企业数：", len(companies))
print(companies[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 任务四：Company"下一页"分页

页面底部分页导航结构（截图 3：`images/03_company_next.png`）：

```html
<nav class="pagination" aria-label="企业分页">
  <span>上一页</span>            <!-- 第 1 页时"上一页"是 span，之后是 a -->
  <b>1</b>
  <a href="/companies?page=2">下一页</a>   <!-- "下一页"始终是最后一个子元素 -->
</nav>
```

因此 selector 用 `nav.pagination > a:last-child`："下一页"始终是分页导航的
最后一个子元素，第 1 页（无上一页链接）和第 2 页起都成立。

In [12]:
next_link = soup.select_one("nav.pagination > a:last-child")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [13]:
# href 是相对路径，用 urljoin 拼接成完整 URL
next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

https://115.159.159.82/companies?page=2


循环方式：**请求当前页 → 解析本页企业 → 找"下一页" href → urljoin 拼出下一页
URL → 请求下一页**，共循环 3 次。

In [14]:
companies = []
current_url = base_url + "/companies"

for page_no in range(1, 4):
    r = session.get(current_url, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    page_companies = [parse_company(c) for c in soup.select("article.company-card")]
    companies.extend(page_companies)
    print(f"第 {page_no} 页：{len(page_companies)} 条（{page_companies[0]['company_id']}"
          f" ~ {page_companies[-1]['company_id']}）→ 累计 {len(companies)} 条")

    next_link = soup.select_one("nav.pagination > a:last-child")
    if next_link is None:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数:", len(companies))

第 1 页：20 条（C000001 ~ C000020）→ 累计 20 条


第 2 页：20 条（C000021 ~ C000040）→ 累计 40 条
第 3 页：20 条（C000041 ~ C000060）→ 累计 60 条
Company 总数: 60


## 任务五：验证采集结果

### 5.1 数量与 ID 唯一性

In [15]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


### 5.2 Company–Device 一致性

用全部 Device 的 `company_id` 统计每家企业实际设备数，
再与 Company 页面显示的 `device_count` 比较。

In [16]:
device_counts = Counter(
    x["company_id"] for x in devices
)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1
        print(f"不一致: {cid} 页面显示 {expected} 台, API 实际 {actual} 台")

print("Company–Device 不一致企业数:", mismatch_count)

Company–Device 不一致企业数: 0


## 任务六：保存采集结果

In [17]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录


In [18]:
import os

for name in ["data/devices.json", "data/companies.json"]:
    size = os.path.getsize(name)
    print(f"{name}: {size:,} 字节")

data/devices.json: 1,977,612 字节
data/companies.json: 12,001 字节


## 实验结论

1. **静态与动态的处理差异**：静态页面的数据在初始 HTML 中，requests 拿到
   `r.text` 后直接用 BeautifulSoup 解析；动态页面的数据不在 HTML 中，
   必须先用 Network 面板找到返回数据的 API，再用 requests 请求 JSON。
2. **两种分页的差异**：HTML 分页要解析每页底部的"下一页"链接，用 urljoin
   拼接相对路径后逐页跟进，翻几页由链接是否存在决定；API 分页直接修改
   `page` 参数循环请求，总页数由返回的 `total` 和 `page_size` 计算得出。
3. **验证方法**：数量核对（Company 60 条 / Device 5000 条）、
   ID 唯一性检查（`set` 去重计数）、跨数据源一致性检查
   （Device 按 `company_id` 计数与 Company 页面 `device_count` 比对）。